PHASE 5: Scraping from https://www.thehindu.com/

In [2]:
import requests
from bs4 import BeautifulSoup
import random
import time
import logging
from urllib.robotparser import RobotFileParser

#Logging setup
logging.basicConfig(
    filename=r"c:\Users\JAY LODHA\web-scraping-project\logs\phase_5_logs.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Scraper started")

#User-Agent rotation list
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Mozilla/5.0 (X11; Linux x86_64)",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 14_0 like Mac OS X)"
]

#Robots.txt compliance
robots = RobotFileParser()
robots.set_url("https://www.thehindu.com/robots.txt")
robots.read()

if not robots.can_fetch("*", "https://www.thehindu.com/news/national/"):
    logging.warning("Scraping not allowed by robots.txt")
    raise PermissionError("Scraping disallowed by robots.txt")

#Request headers and session
session = requests.Session()
headers = {"User-Agent": random.choice(USER_AGENTS)}

#Retry mechanism
def fetch_url(url, retries=3, delay=2):
    for attempt in range(retries):
        try:
            response = session.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as e:
            logging.error(f"Attempt {attempt+1} failed: {e}")
            time.sleep(delay)
    return None

#Scrape The Hindu news
url = "https://www.thehindu.com/news/national/"
response = fetch_url(url)

if response:
    soup = BeautifulSoup(response.text, "html.parser")
    articles = soup.find_all("h3", class_="title big")

    data = []
    for a in articles:
        title = a.get_text(strip=True)
        link = a.find("a")["href"] if a.find("a") else None
        data.append({"Title": title, "Link": link})
        logging.info(f"Scraped: {title}")

    #Save results
    import pandas as pd
    df = pd.DataFrame(data)
    df.to_csv("data/the_hindu_news.csv", index=False)
    logging.info("Scraping completed successfully!")

else:
    logging.error("Failed to fetch The Hindu page after retries.")


PermissionError: Scraping disallowed by robots.txt